# Preparación de datos para predicción (horizonte h=6/12/24)

**Proyecto de Aprendizaje Automático en Series Temporales y Flujos de Datos**

Este cuaderno construye el dataset de *features* que usarán los notebooks de modelado, a partir de `../data/processed/beijing_data_preparado.csv` (salida de `02_preparacion_datos.ipynb`). El horizonte de pronóstico queda fijado en **h=6/12/24 horas**.

In [26]:
NUMERO_DE_HORAS=6

## 1. Carga del dataset preparado

In [27]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/beijing_data_preparado.csv', index_col='datetime', parse_dates=True)
df = df.asfreq('h')
df.drop(columns=["pm2.5_imputado"],inplace=True) #quito la columna de pm2.5_imputado, no la uso para predecir

print("Observaciones:", len(df))
df.head()

Observaciones: 43824


,pm2.5,DEWP,TEMP,PRES,Iws,Is,Ir,viento_NE,viento_NW,viento_SE,viento_cv
datetime,,,,,,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,1.79,0,0,False,True,False,False
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,4.92,0,0,False,True,False,False
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,6.71,0,0,False,True,False,False
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,9.84,0,0,False,True,False,False
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,12.97,0,0,False,True,False,False


## 2. Variables de calendario (codificación cíclica seno/coseno)

Se codifican hora del día, día de la semana y mes como pares seno/coseno para que el modelo vea la naturaleza cíclica de estas variables.

IMPORTANTE: Las variables de calendario se conocen en todo momento y no serán desplazadas con lags.

In [28]:
hora = df.index.hour
dia = df.index.dayofweek
mes = df.index.month

df['cal_sen_hora'] = np.sin(2 * np.pi * hora / 24)
df['cal_cos_hora'] = np.cos(2 * np.pi * hora / 24)

df['cal_sen_dia'] = np.sin(2 * np.pi * dia / 7)
df['cal_cos_dia'] = np.cos(2 * np.pi * dia / 7)

df['cal_sen_mes'] = np.sin(2 * np.pi * (mes - 1) / 12)
df['cal_cos_mes'] = np.cos(2 * np.pi * (mes - 1) / 12)

df[['cal_sen_hora', 'cal_cos_hora', 'cal_sen_dia', 'cal_cos_dia', 'cal_sen_mes', 'cal_cos_mes']].head()

,cal_sen_hora,cal_cos_hora,cal_sen_dia,cal_cos_dia,cal_sen_mes,cal_cos_mes
datetime,,,,,,
2010-01-01 00:00:00,0.000000,1.000000,-0.433884,-0.900969,0.0,1.0
2010-01-01 01:00:00,0.258819,0.965926,-0.433884,-0.900969,0.0,1.0
2010-01-01 02:00:00,0.500000,0.866025,-0.433884,-0.900969,0.0,1.0
2010-01-01 03:00:00,0.707107,0.707107,-0.433884,-0.900969,0.0,1.0
2010-01-01 04:00:00,0.866025,0.500000,-0.433884,-0.900969,0.0,1.0


## 3 - Creación de lags a 6 horas

Para toda exógena que no sea de calendario, la desplazo con un lag de 6 horas. En este caso, para cada valor a predecir (variable y), la información de la exógena que tengo es lo que sucedió 6 horas antes.

Se creará por tanto las variables "var_obs" (variable observada) con dicho lag.

In [29]:
cols_exog_no_cal = [c for c in df.columns if c != "pm2.5" and not c.startswith('cal')]

obs = df[cols_exog_no_cal].shift(NUMERO_DE_HORAS).add_suffix(f'_obs_{NUMERO_DE_HORAS}')
df = pd.concat([df, obs], axis=1)

df.filter(like='_obs').head(8)

,DEWP_obs_6,TEMP_obs_6,PRES_obs_6,Iws_obs_6,Is_obs_6,Ir_obs_6,viento_NE_obs_6,viento_NW_obs_6,viento_SE_obs_6,viento_cv_obs_6
datetime,,,,,,,,,,
2010-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 05:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 06:00:00,-21.0,-11.0,1021.0,1.79,0.0,0.0,False,True,False,False
2010-01-01 07:00:00,-21.0,-12.0,1020.0,4.92,0.0,0.0,False,True,False,False


In [30]:
cols_predictivas = [c for c in df.columns if c.startswith("cal") or "obs" in c]
cols_predictivas

['cal_sen_hora',
 'cal_cos_hora',
 'cal_sen_dia',
 'cal_cos_dia',
 'cal_sen_mes',
 'cal_cos_mes',
 'DEWP_obs_6',
 'TEMP_obs_6',
 'PRES_obs_6',
 'Iws_obs_6',
 'Is_obs_6',
 'Ir_obs_6',
 'viento_NE_obs_6',
 'viento_NW_obs_6',
 'viento_SE_obs_6',
 'viento_cv_obs_6']

In [31]:
df_predictivo = df[["pm2.5"]+cols_predictivas]
df_predictivo.head()

,pm2.5,cal_sen_hora,cal_cos_hora,cal_sen_dia,cal_cos_dia,cal_sen_mes,cal_cos_mes,DEWP_obs_6,TEMP_obs_6,PRES_obs_6,Iws_obs_6,Is_obs_6,Ir_obs_6,viento_NE_obs_6,viento_NW_obs_6,viento_SE_obs_6,viento_cv_obs_6
datetime,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,129.0,0.000000,1.000000,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 01:00:00,129.0,0.258819,0.965926,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 02:00:00,129.0,0.500000,0.866025,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 03:00:00,129.0,0.707107,0.707107,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-01 04:00:00,129.0,0.866025,0.500000,-0.433884,-0.900969,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Finalmente nos quedamos con las filas que no contienen valores nulos (las 6 primeras se eliminan y se eliminan el resto de NaNs de pm2.5)

In [32]:
df_predictivo.shape

(43824, 17)

In [33]:
df_predictivo.isna().sum()

pm2.5              1463
cal_sen_hora          0
cal_cos_hora          0
cal_sen_dia           0
cal_cos_dia           0
cal_sen_mes           0
cal_cos_mes           0
DEWP_obs_6            6
TEMP_obs_6            6
PRES_obs_6            6
Iws_obs_6             6
Is_obs_6              6
Ir_obs_6              6
viento_NE_obs_6       6
viento_NW_obs_6       6
viento_SE_obs_6       6
viento_cv_obs_6       6
dtype: int64

In [34]:
cols_na = list(df_predictivo.filter(like='_obs').columns)
df_predictivo = df_predictivo.dropna(subset=cols_na)

In [35]:
df_predictivo.isna().sum()

pm2.5              1463
cal_sen_hora          0
cal_cos_hora          0
cal_sen_dia           0
cal_cos_dia           0
cal_sen_mes           0
cal_cos_mes           0
DEWP_obs_6            0
TEMP_obs_6            0
PRES_obs_6            0
Iws_obs_6             0
Is_obs_6              0
Ir_obs_6              0
viento_NE_obs_6       0
viento_NW_obs_6       0
viento_SE_obs_6       0
viento_cv_obs_6       0
dtype: int64

In [36]:
df_predictivo.to_csv(f'../data/processed/beijing_data_huecos_{NUMERO_DE_HORAS}h.csv')

In [37]:
df_predictivo = df_predictivo.dropna(subset=["pm2.5"])

In [38]:
df_predictivo.head()

,pm2.5,cal_sen_hora,cal_cos_hora,cal_sen_dia,cal_cos_dia,cal_sen_mes,cal_cos_mes,DEWP_obs_6,TEMP_obs_6,PRES_obs_6,Iws_obs_6,Is_obs_6,Ir_obs_6,viento_NE_obs_6,viento_NW_obs_6,viento_SE_obs_6,viento_cv_obs_6
datetime,,,,,,,,,,,,,,,,,
2010-01-01 06:00:00,129.0,1.000000,6.123234e-17,-0.433884,-0.900969,0.0,1.0,-21.0,-11.0,1021.0,1.79,0.0,0.0,False,True,False,False
2010-01-01 07:00:00,129.0,0.965926,-2.588190e-01,-0.433884,-0.900969,0.0,1.0,-21.0,-12.0,1020.0,4.92,0.0,0.0,False,True,False,False
2010-01-01 08:00:00,129.0,0.866025,-5.000000e-01,-0.433884,-0.900969,0.0,1.0,-21.0,-11.0,1019.0,6.71,0.0,0.0,False,True,False,False
2010-01-01 09:00:00,129.0,0.707107,-7.071068e-01,-0.433884,-0.900969,0.0,1.0,-21.0,-14.0,1019.0,9.84,0.0,0.0,False,True,False,False
2010-01-01 10:00:00,129.0,0.500000,-8.660254e-01,-0.433884,-0.900969,0.0,1.0,-20.0,-12.0,1018.0,12.97,0.0,0.0,False,True,False,False


In [39]:
df_predictivo.to_csv(f'../data/processed/beijing_data_{NUMERO_DE_HORAS}h.csv')